In [2]:
!pip uninstall -y transformers
!pip install -U transformers accelerate bitsandbytes sentencepiece

Found existing installation: transformers 5.10.1
Uninstalling transformers-5.10.1:
  Successfully uninstalled transformers-5.10.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 133.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.3 MB/s eta 0:00:00
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.13.0
    Uninstalling accelerate-1.13.0:
      Successfully uninstalled accelerate-1.13.0


In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import pandas as pd
import re

In [4]:
def extract_final_num(text):
    match = re.search(r"Final\s+Answer:\s*(-?\d+\.?\d*)", text, re.IGNORECASE)
    if match:
        return match.group(1)
    nums = re.findall(r'-?\d+\.?\d*', text)
    if nums:
        return nums[-1]
    return None

In [6]:
test_df = pd.read_csv("unified_svamp_test.csv").head(50)

In [7]:
import os
os.environ['LD_LIBRARY_PATH'] = os.environ.get('LD_LIBRARY_PATH', '') + ':/usr/local/cuda-13.0/lib64'

In [8]:
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get('HF_TOKEN')
login(hf_token)
print("Authenticated successfully!")

Authenticated successfully!


In [9]:
model_id = "Qwen/Qwen2.5-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16   # bfloat16 for Qwen, not float16
)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_id)

In [ ]:
print("Downloading the model")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

In [12]:
results = []
correct_count = 0
total_count = len(test_df)

In [14]:
accuracy = (correct_count / total_count) * 100
print(f"FINAL BASELINE ACCURACY: {accuracy:.2f}% ({correct_count}/{total_count})")

FINAL BASELINE ACCURACY: 46.00% (23/50)
